In [ ]:
!python.exe -m pip install --upgrade pip

In [ ]:
!pip install distilabel[hf-inference-endpoints] --upgrade

In [ ]:
!pip install distilabel --upgrade

In [ ]:
!pip install 'distilabel[anthropic]'

In [ ]:
!pip install 'distilabel[instructor]'

In [ ]:
!pip install --upgrade distilabel anthropic pydantic

In [ ]:
import json, os

sample_data = [
    {
        "instruction": "Help me set up my daily to-do list!",
        "input": "Setting up your daily to-do list...",
        "output": """```python
tasks = []
while True:
    task = input('Enter a task or type \'done\' to finish: ')
    if task == 'done': break
    tasks.append(task)
print(f'Your to-do list for today: {tasks}')
```""",
        "text": """Help me set up my daily to-do list! Setting up your daily to-do list... ```python
tasks = []
while True:
    task = input('Enter a task or type \'done\' to finish: ')
    if task == 'done': break
    tasks.append(task)
print(f'Your to-do list for today: {tasks}')
```"""
    },
    {
        "instruction": "Create a shopping list based on my inputs!",
        "input": "Creating a shopping list...",
        "output": """```python
shopping_list = {}
while True:
    item = input('Enter an item or type \'done\' to finish: ')
    if item == 'done': break
    quantity = input(f'Enter the quantity for {item}: ')
    shopping_list[item] = quantity
print(f'Your shopping list: {shopping_list}')
```""",
        "text": """Create a shopping list based on my inputs! Creating a shopping list... ```python
shopping_list = {}
while True:
    item = input('Enter an item or type \'done\' to finish: ')
    if item == 'done': break
    quantity = input(f'Enter the quantity for {item}: ')
    shopping_list[item] = quantity
print(f'Your shopping list: {shopping_list}')
```"""
    }
]

with open("/content/sample_data/data.json", "w") as f:
    json.dump(sample_data, f, indent=4)

print("data.json file created successfully.")

In [ ]:
import json, os
from distilabel.pipeline import Pipeline
from distilabel.steps.tasks import TextGeneration
from distilabel.models.llms import AnthropicLLM

with open("/content/sample_data/data.json", "r") as h:
    local_data = json.load(h)

with Pipeline() as pipeline:
    task = TextGeneration(
        name="text-generation-step",
        llm = AnthropicLLM(
        api_key=os.getenv("ANTHROPIC_API_KEY"),
        model="claude-3-haiku-20240307"
    ),
    )

if __name__ == "__main__":
    print("Running pipeline...")
    result1 = pipeline.run(dataset=local_data,use_cache=False)

    print(type(result1))
    print(result1)
    generated_data = result1["default"]["train"]
    print("Generated_data",generated_data)
    for i, item in enumerate(generated_data):
        print(f"\n--- Result {i + 1} ---")
        print(f"Instruction: {item.get('instruction')}")
        print(f"Input: {item.get('input')}")
        print(f"Output: {item.get('output')}")

In [ ]:
generated_data[0]['output']

In [ ]:
print(result1)

In [ ]:
generated_data = result1["default"]["train"]

In [ ]:
generated_data[0]

In [ ]:
generated_data['instruction']

In [ ]:
generated_data['generation']

In [ ]:
generated_data[0]['generation']

In [ ]:
print("\nNow you can enter your own prompt for generation!")

user_instruction = input("Instruction: ")

user_dataset = {
    "train": [
        {
            "instruction": user_instruction,
        }
    ]
}

with Pipeline() as pipeline:
    task = TextGeneration(
        name="text-generation-step",
        llm=AnthropicLLM(
            api_key=os.getenv("ANTHROPIC_API_KEY"),
            model="claude-3-haiku-20240307"
        )
    )
result = pipeline.run(dataset=[{
    "instruction": user_instruction
}], use_cache=False)

generated_response = result["default"]["train"][0]["generation"]
print("\nGenerated Output:")
print(generated_response)